# Nonlinear Model Comparison

This notebook compares the selected Ridge baseline with Random Forest models using the prepared player-season data, chronological splits, and selected features from the Ridge notebook.

Most reusable logic now lives in `src/prem_valuation/`, so this notebook focuses on the experiment flow and the resulting 2025/26 ranking outputs.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

import pandas as pd

from prem_valuation.data import (
    PROCESSED_DIR,
    load_appearances,
    load_competitions,
    load_games,
    load_interim_tables,
    load_players,
    load_valuations,
)
from prem_valuation.features import (
    HISTORY_FEATURES,
    HISTORY_NUMERIC_FEATURES,
    TEAM_CONTEXT_FEATURES_ALL,
    TEAM_CONTEXT_NUMERIC_FEATURES,
    WEIGHTED_HISTORY_FEATURES_ALL,
    WEIGHTED_HISTORY_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    SELECTED_FEATURES,
    SELECTED_NUMERIC_FEATURES,
    TARGET,
    add_engineered_features,
    add_latest_history_to_scoring,
    add_latest_weighted_history_to_scoring,
    add_preseason_market_values,
    add_previous_season_features,
    add_team_context_features,
    add_weighted_previous_history,
    build_player_season_clubs,
    build_team_season_context,
    build_weighted_appearance_history,
)
from prem_valuation.modeling import (
    add_log_value_change_target,
    evaluate_predictions,
    make_random_forest_model,
    make_ridge_model,
    predict_non_negative,
    predict_value_from_log_change,
    walk_forward_evaluate,
    walk_forward_evaluate_log_change,
)
from prem_valuation.rankings import (
    attach_current_player_values,
    attach_season_club_metadata,
    build_season_club_lookup,
    build_scoring_results,
    make_ranking_tables,
    save_ranking_outputs,
)


## Load prepared data

In [2]:
model_data, scoring_data = load_interim_tables()
valuations = load_valuations()

model_data = add_engineered_features(model_data)
scoring_data = add_engineered_features(scoring_data)

model_data.shape, scoring_data.shape

((6301, 24), (537, 21))

## Chronological split

In [3]:
train_data = model_data.loc[model_data["season"].between(2016, 2022)].copy()
validation_data = model_data.loc[model_data["season"].eq(2023)].copy()
test_data = model_data.loc[model_data["season"].eq(2024)].copy()

for name, dataset in {
    "train": train_data,
    "validation": validation_data,
    "test": test_data,
    "scoring": scoring_data,
}.items():
    print(name, dataset.shape, sorted(dataset["season"].unique()))

train (3476, 24) [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
validation (534, 24) [np.int64(2023)]
test (477, 24) [np.int64(2024)]
scoring (537, 21) [np.int64(2025)]


## Model builders

In [4]:
ridge_builder = lambda: make_ridge_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
    alpha=100,
)

forest_builder = lambda: make_random_forest_model(
    SELECTED_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

history_forest_builder = lambda: make_random_forest_model(
    HISTORY_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

weighted_history_forest_builder = lambda: make_random_forest_model(
    WEIGHTED_HISTORY_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

team_context_forest_builder = lambda: make_random_forest_model(
    TEAM_CONTEXT_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)


## Baseline nonlinear comparison

Compare the frozen Ridge baseline against a plain Random Forest using walk-forward validation.

In [5]:
ridge_cv = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    ridge_builder,
)
forest_cv = walk_forward_evaluate(
    model_data,
    SELECTED_FEATURES,
    TARGET,
    forest_builder,
)

model_comparison = pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
}).T

model_comparison

,mae,rmse,r2
ridge,9.289876e+06,1.411000e+07,0.497488
random_forest,8.785972e+06,1.379553e+07,0.519412


In [6]:
fold_comparison = ridge_cv.merge(
    forest_cv,
    on="validation_season",
    suffixes=("_ridge", "_forest"),
)
fold_comparison["mae_improvement"] = (
    fold_comparison["mae_ridge"] - fold_comparison["mae_forest"]
)

fold_comparison

,validation_season,mae_ridge,rmse_ridge,r2_ridge,mae_forest,rmse_forest,r2_forest,mae_improvement
0,2019,9.067193e+06,1.427333e+07,0.488337,8.317015e+06,1.336068e+07,0.551677,750177.762444
1,2020,9.245061e+06,1.343545e+07,0.487530,8.608435e+06,1.304619e+07,0.516795,636625.718159
2,2021,8.414830e+06,1.243830e+07,0.532694,8.035258e+06,1.249307e+07,0.528570,379571.565190
3,2022,9.255718e+06,1.444655e+07,0.484836,8.686980e+06,1.436498e+07,0.490637,568737.351968
4,2023,1.046658e+07,1.595639e+07,0.494044,1.028217e+07,1.571272e+07,0.509379,184405.510213


## Final Random Forest test evaluation

Random Forest was selected using walk-forward validation only. We now evaluate it once on the untouched 2024/25 test season.

In [7]:
development_data = pd.concat(
    [train_data, validation_data],
    ignore_index=True,
)

final_forest_model = forest_builder()
final_forest_model.fit(
    development_data[SELECTED_FEATURES],
    development_data[TARGET],
)

test_predictions = predict_non_negative(
    final_forest_model,
    test_data[SELECTED_FEATURES],
)

pd.Series(
    evaluate_predictions(test_data[TARGET], test_predictions),
    name="2024/25 final test",
)

mae     9.458311e+06
rmse    1.581239e+07
r2      4.853420e-01
Name: 2024/25 final test, dtype: float64

## Random Forest residual analysis

Inspect the model's biggest misses to see whether the nonlinear model reduces the Ridge baseline's tendency to compress elite players toward the average.

In [8]:
test_results = test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        TARGET,
    ]
].copy()

test_results["predicted_value"] = test_predictions
test_results["residual"] = test_results[TARGET] - test_results["predicted_value"]
test_results["absolute_error"] = test_results["residual"].abs()

test_results.sort_values("absolute_error", ascending=False).head(15)

,season,player_id,player_name,position,sub_position,age_at_season_end,minutes_played,market_value_in_eur,predicted_value,residual,absolute_error
5911,2024,357565,Rodri,Midfield,Defensive Midfield,28.922656,73,110000000.0,4.289239e+06,1.057108e+08,1.057108e+08
5867,2024,433177,Bukayo Saka,Attack,Right Winger,23.718001,1736,150000000.0,5.080691e+07,9.919309e+07,9.919309e+07
5899,2024,418560,Erling Haaland,Attack,Centre-Forward,24.843258,2743,180000000.0,8.744035e+07,9.255965e+07,9.255965e+07
5925,2024,357662,Declan Rice,Midfield,Central Midfield,26.360027,2833,120000000.0,3.690773e+07,8.309227e+07,8.309227e+07
6071,2024,534033,Alexis Mac Allister,Midfield,Central Midfield,26.417522,2609,100000000.0,3.139148e+07,6.860852e+07,6.860852e+07
5874,2024,445939,Omar Marmoush,Attack,Centre-Forward,26.294319,1183,75000000.0,2.560253e+07,4.939747e+07,4.939747e+07
5889,2024,435338,Gabriel,Defender,Centre-Back,27.430527,2365,75000000.0,2.621263e+07,4.878737e+07,4.878737e+07
5999,2024,687626,Moisés Caicedo,Midfield,Defensive Midfield,23.559206,3356,90000000.0,4.180377e+07,4.819623e+07,4.819623e+07
6121,2024,258004,Rúben Dias,Defender,Centre-Back,28.030116,2269,65000000.0,1.725504e+07,4.774496e+07,4.774496e+07
6040,2024,495666,William Saliba,Defender,Centre-Back,24.169747,3041,80000000.0,3.327902e+07,4.672098e+07,4.672098e+07


## Add historical features

The first Random Forest only sees the current season. We now add previous-season performance and previous market value so the model has memory of player reputation and recent history.

In [9]:
model_data_with_history = add_previous_season_features(model_data)
model_data_with_history = add_preseason_market_values(
    model_data_with_history,
    valuations,
)

appearances = load_appearances()
games = load_games()
competitions = load_competitions()

appearance_history = build_weighted_appearance_history(
    appearances,
    games,
    competitions,
)
model_data_with_weighted_history = add_weighted_previous_history(
    model_data_with_history,
    appearance_history,
)

team_context = build_team_season_context(games)
player_season_clubs = build_player_season_clubs(appearances, games)
model_data_with_team_context = add_team_context_features(
    model_data_with_weighted_history,
    player_season_clubs,
    team_context,
)

model_data_with_team_context[
    [
        "player_name",
        "season",
        "minutes_played",
        "previous_known_market_value_in_eur",
        "previous_weighted_goals",
        "team_points",
        "team_final_position",
        "team_won_league",
        "player_minutes_share",
        TARGET,
    ]
].head(10)


,player_name,season,minutes_played,previous_known_market_value_in_eur,previous_weighted_goals,team_points,team_final_position,team_won_league,player_minutes_share,market_value_in_eur
0,Alou Diarra,2012,139,6000000.0,NaN,46,10,0,0.040643,1250000.0
1,Magaye Gueye,2012,49,2500000.0,NaN,63,6,0,0.014327,2000000.0
2,André Santos,2012,395,7500000.0,NaN,73,4,0,0.115497,2500000.0
3,Lucas Piazón,2012,16,7500000.0,NaN,75,3,0,0.004678,5000000.0
4,Anton Ferdinand,2012,914,3500000.0,NaN,25,20,0,0.267251,2000000.0
5,Maurice Edu,2012,10,3000000.0,NaN,42,13,0,0.002924,2300000.0
6,Raúl Meireles,2012,100,15000000.0,NaN,75,3,0,0.029240,12000000.0
7,Pajtim Kasami,2012,22,2500000.0,NaN,43,12,0,0.006433,2000000.0
8,Alejandro Faurlín,2012,782,4500000.0,NaN,25,20,0,0.228655,3300000.0
9,Nigel de Jong,2012,90,13000000.0,NaN,78,2,0,0.026316,7000000.0


## History model comparison

Compare the plain Random Forest against the history-aware Random Forest using the same walk-forward folds.

In [10]:
history_forest_cv = walk_forward_evaluate(
    model_data_with_history,
    HISTORY_FEATURES,
    TARGET,
    history_forest_builder,
)
weighted_history_forest_cv = walk_forward_evaluate(
    model_data_with_weighted_history,
    WEIGHTED_HISTORY_FEATURES_ALL,
    TARGET,
    weighted_history_forest_builder,
)
team_context_forest_cv = walk_forward_evaluate(
    model_data_with_team_context,
    TEAM_CONTEXT_FEATURES_ALL,
    TARGET,
    team_context_forest_builder,
)

pd.DataFrame({
    "ridge": ridge_cv[["mae", "rmse", "r2"]].mean(),
    "random_forest": forest_cv[["mae", "rmse", "r2"]].mean(),
    "history_random_forest": history_forest_cv[["mae", "rmse", "r2"]].mean(),
    "weighted_history_random_forest": weighted_history_forest_cv[["mae", "rmse", "r2"]].mean(),
    "team_context_random_forest": team_context_forest_cv[["mae", "rmse", "r2"]].mean(),
}).T


,mae,rmse,r2
ridge,9.289876e+06,1.411000e+07,0.497488
random_forest,8.785972e+06,1.379553e+07,0.519412
history_random_forest,5.057773e+06,8.233374e+06,0.823524
weighted_history_random_forest,5.063675e+06,8.296923e+06,0.820843
team_context_random_forest,4.757691e+06,7.935216e+06,0.835692


In [11]:
team_context_fold_comparison = weighted_history_forest_cv.merge(
    team_context_forest_cv,
    on="validation_season",
    suffixes=("_weighted_history_forest", "_team_context_forest"),
)
team_context_fold_comparison["mae_improvement"] = (
    team_context_fold_comparison["mae_weighted_history_forest"]
    - team_context_fold_comparison["mae_team_context_forest"]
)

team_context_fold_comparison


,validation_season,mae_weighted_history_forest,rmse_weighted_history_forest,r2_weighted_history_forest,mae_team_context_forest,rmse_team_context_forest,r2_team_context_forest,mae_improvement
0,2019,6.557902e+06,1.011215e+07,0.743185,6.149892e+06,9.662905e+06,0.765497,408009.571963
1,2020,4.624956e+06,7.700448e+06,0.831657,4.481788e+06,7.478763e+06,0.841210,143167.625928
2,2021,3.822930e+06,6.145038e+06,0.885941,3.502748e+06,5.733607e+06,0.900703,320181.204627
3,2022,5.684981e+06,9.847688e+06,0.760621,5.409414e+06,9.514075e+06,0.776565,275566.837614
4,2023,4.627609e+06,7.679293e+06,0.882811,4.244613e+06,7.286730e+06,0.894486,382995.861691


## Valuation update model

Absolute-value models tend to compress elite players toward the average. A valuation update model predicts the percentage change from the latest known preseason market value, then converts that predicted change back into euros.

In [12]:
model_data_with_team_context = add_log_value_change_target(
    model_data_with_team_context,
    target=TARGET,
)

log_change_forest_builder = lambda: make_random_forest_model(
    TEAM_CONTEXT_NUMERIC_FEATURES,
    SELECTED_CATEGORICAL_FEATURES,
)

log_change_cv = walk_forward_evaluate_log_change(
    model_data_with_team_context,
    TEAM_CONTEXT_FEATURES_ALL,
    TARGET,
    "log_value_change",
    log_change_forest_builder,
)

pd.DataFrame({
    "team_context_absolute": team_context_forest_cv[["mae", "rmse", "r2"]].mean(),
    "team_context_log_change": log_change_cv[["mae", "rmse", "r2"]].mean(),
}).T


,mae,rmse,r2
team_context_absolute,4.757691e+06,7.935216e+06,0.835692
team_context_log_change,5.043146e+06,8.858970e+06,0.785995


In [13]:
log_change_cv

,validation_season,mae,rmse,r2,rows
0,2019,7.960299e+06,1.428831e+07,0.507794,451
1,2020,4.654694e+06,8.313061e+06,0.807983,474
2,2021,3.387959e+06,6.064309e+06,0.889435,488
3,2022,4.910828e+06,8.449152e+06,0.825929,514
4,2023,4.301949e+06,7.180020e+06,0.898834,520


## Final weighted history Random Forest test evaluation

The weighted history Random Forest adds previous all-competition production with simple competition weights. We evaluate it once on the untouched 2024/25 test season, then later train a production scoring model on all labelled seasons.

In [14]:
development_data_with_team_context = model_data_with_team_context.loc[
    model_data_with_team_context["season"].between(2016, 2023)
].copy()
test_data_with_team_context = model_data_with_team_context.loc[
    model_data_with_team_context["season"].eq(2024)
].copy()

final_team_context_forest_model = team_context_forest_builder()
final_team_context_forest_model.fit(
    development_data_with_team_context[TEAM_CONTEXT_FEATURES_ALL],
    development_data_with_team_context[TARGET],
)
team_context_test_predictions = predict_non_negative(
    final_team_context_forest_model,
    test_data_with_team_context[TEAM_CONTEXT_FEATURES_ALL],
)

final_log_change_model = log_change_forest_builder()
log_change_training_data = development_data_with_team_context.loc[
    development_data_with_team_context["log_value_change"].notna()
    & development_data_with_team_context["previous_known_market_value_in_eur"].gt(0)
].copy()
final_log_change_model.fit(
    log_change_training_data[TEAM_CONTEXT_FEATURES_ALL],
    log_change_training_data["log_value_change"],
)
log_change_test_data = test_data_with_team_context.loc[
    test_data_with_team_context["previous_known_market_value_in_eur"].gt(0)
].copy()
log_change_test_predictions = predict_value_from_log_change(
    final_log_change_model,
    log_change_test_data[TEAM_CONTEXT_FEATURES_ALL],
)

pd.DataFrame({
    "team_context_absolute": evaluate_predictions(
        test_data_with_team_context[TARGET],
        team_context_test_predictions,
    ),
    "team_context_log_change": evaluate_predictions(
        log_change_test_data[TARGET],
        log_change_test_predictions,
    ),
}).T


,mae,rmse,r2
team_context_absolute,4.682582e+06,7.966765e+06,0.869356
team_context_log_change,4.645308e+06,7.796095e+06,0.876415


In [15]:
log_change_test_results = log_change_test_data[
    [
        "season",
        "player_id",
        "player_name",
        "position",
        "sub_position",
        "age_at_season_end",
        "minutes_played",
        "previous_known_market_value_in_eur",
        "previous_weighted_goals",
        "team_points",
        "team_final_position",
        "team_won_league",
        "player_minutes_share",
        TARGET,
        "previous_market_value_in_eur",
    ]
].copy()

log_change_test_results["predicted_value"] = log_change_test_predictions
log_change_test_results["residual"] = (
    log_change_test_results[TARGET]
    - log_change_test_results["predicted_value"]
)
log_change_test_results["absolute_error"] = log_change_test_results["residual"].abs()

log_change_test_results.sort_values("absolute_error", ascending=False).head(15)


,season,player_id,player_name,position,sub_position,age_at_season_end,minutes_played,previous_known_market_value_in_eur,previous_weighted_goals,team_points,team_final_position,team_won_league,player_minutes_share,market_value_in_eur,previous_market_value_in_eur,predicted_value,residual,absolute_error
5874,2024,445939,Omar Marmoush,Attack,Centre-Forward,26.294319,1183,22000000.0,11.2,71,3,0,0.345906,75000000.0,NaN,2.587215e+07,4.912785e+07,4.912785e+07
5878,2024,890721,Myles Lewis-Skelly,Defender,Left-Back,18.661191,1371,2000000.0,NaN,74,2,0,0.400877,45000000.0,NaN,6.000000e+06,3.900000e+07,3.900000e+07
5859,2024,349066,Alexander Isak,Attack,Centre-Forward,25.675565,2774,75000000.0,23.0,66,5,0,0.811111,120000000.0,75000000.0,8.596846e+07,3.403154e+07,3.403154e+07
5962,2024,890719,Ethan Nwaneri,Midfield,Attacking Midfield,18.179329,889,8000000.0,0.0,74,2,0,0.259942,55000000.0,NaN,2.205036e+07,3.294964e+07,3.294964e+07
6290,2024,315969,Scott McTominay,Midfield,Central Midfield,28.459959,17,32000000.0,9.0,42,15,0,0.004971,50000000.0,32000000.0,1.808661e+07,3.191339e+07,3.191339e+07
6016,2024,610442,Rasmus Højlund,Attack,Centre-Forward,22.302533,2014,65000000.0,15.5,42,15,0,0.588889,35000000.0,65000000.0,6.496030e+07,-2.996030e+07,2.996030e+07
5883,2024,444523,Lucas Paquetá,Midfield,Attacking Midfield,27.742642,2385,65000000.0,7.2,43,14,0,0.697368,28000000.0,65000000.0,5.772762e+07,-2.972762e+07,2.972762e+07
5911,2024,357565,Rodri,Midfield,Defensive Midfield,28.922656,73,120000000.0,9.0,71,3,0,0.021345,110000000.0,120000000.0,8.108303e+07,2.891697e+07,2.891697e+07
6078,2024,763079,Abdukodir Khusanov,Defender,Centre-Back,21.234771,504,2500000.0,0.0,71,3,0,0.147368,35000000.0,NaN,7.500000e+06,2.750000e+07,2.750000e+07
5876,2024,466805,Nico González,Midfield,Defensive Midfield,23.389459,764,15000000.0,1.4,71,3,0,0.223392,40000000.0,NaN,1.629382e+07,2.370618e+07,2.370618e+07


In [16]:
log_change_test_results["has_previous_market_value"] = (
    log_change_test_results["previous_known_market_value_in_eur"].notna()
)

log_change_test_results.groupby("has_previous_market_value").agg(
    players=("player_id", "count"),
    actual_mean=(TARGET, "mean"),
    predicted_mean=("predicted_value", "mean"),
    mae=("absolute_error", "mean"),
    mean_residual=("residual", "mean"),
)

,players,actual_mean,predicted_mean,mae,mean_residual
has_previous_market_value,,,,,
True,459,2.199592e+07,2.197273e+07,4.645308e+06,23182.578409


## Score 2025/26 players

Apply the selected history Random Forest model to the 2025/26 scoring population. The prediction is an expected Transfermarkt-style value; the valuation gap is the model prediction minus the current Transfermarkt value.

In [17]:
players = load_players()
season_club_lookup = build_season_club_lookup(
    appearances,
    games,
    players,
    season=2025,
)

scoring_data_with_history = add_latest_history_to_scoring(
    scoring_data,
    model_data,
    history_season=2024,
)
scoring_data_with_history = add_latest_weighted_history_to_scoring(
    scoring_data_with_history,
    appearance_history,
    history_season=2024,
)
scoring_data_with_history = add_preseason_market_values(
    scoring_data_with_history,
    valuations,
)
scoring_data_with_history = add_team_context_features(
    scoring_data_with_history,
    player_season_clubs,
    team_context,
)
scoring_data_with_history = attach_season_club_metadata(
    scoring_data_with_history,
    season_club_lookup,
)
scoring_data_with_history = attach_current_player_values(
    scoring_data_with_history,
    players,
)

scoring_data_with_history[
    [
        "player_name",
        "season_club_name",
        "current_club_name",
        "minutes_played",
        "previous_weighted_goals",
        "previous_known_market_value_in_eur",
        "team_points",
        "team_final_position",
        "team_won_league",
        "player_minutes_share",
        "current_market_value_in_eur",
    ]
].head(10)


,player_name,season_club_name,current_club_name,minutes_played,previous_weighted_goals,previous_known_market_value_in_eur,team_points,team_final_position,team_won_league,player_minutes_share,current_market_value_in_eur
0,James Milner,Brighton & Hove Albion,Brighton & Hove Albion,778,0.0,1000000.0,53,8,0,0.227485,750000.0
1,Casemiro,Manchester United,Manchester United,2589,2.6,10000000.0,71,3,0,0.757018,8000000.0
2,Mateo Kovacic,Manchester City,Manchester City,126,7.0,20000000.0,78,2,0,0.036842,15000000.0
3,Jordan Henderson,Brentford FC,Brentford FC,1921,0.7,2500000.0,53,9,0,0.561696,2000000.0
4,Adam Smith,AFC Bournemouth,AFC Bournemouth,1080,0.0,750000.0,57,6,0,0.315789,500000.0
5,Ashley Barnes,Burnley FC,Burnley FC,41,NaN,1000000.0,22,19,0,0.011988,1000000.0
6,Danny Welbeck,Brighton & Hove Albion,Brighton & Hove Albion,2266,10.5,5000000.0,53,8,0,0.662573,4000000.0
7,Séamus Coleman,Everton FC,Everton FC,20,0.0,500000.0,49,13,0,0.005848,300000.0
8,Bernd Leno,Fulham FC,Fulham FC,3420,0.0,10000000.0,52,11,0,1.000000,8000000.0
9,Martin Dúbravka,Burnley FC,Burnley FC,3150,0.0,1000000.0,22,19,0,0.921053,750000.0


In [18]:
production_training_data = model_data_with_team_context.loc[
    model_data_with_team_context["season"].between(2016, 2024)
].copy()
production_log_change_training_data = production_training_data.loc[
    production_training_data["log_value_change"].notna()
    & production_training_data["previous_known_market_value_in_eur"].gt(0)
].copy()

production_log_change_model = log_change_forest_builder()
production_log_change_model.fit(
    production_log_change_training_data[TEAM_CONTEXT_FEATURES_ALL],
    production_log_change_training_data["log_value_change"],
)

scoring_predictions = predict_value_from_log_change(
    production_log_change_model,
    scoring_data_with_history[TEAM_CONTEXT_FEATURES_ALL],
)
scoring_results = build_scoring_results(
    scoring_data_with_history,
    scoring_predictions,
)

scoring_results[
    [
        "player_name",
        "season_club_name",
        "current_club_name",
        "position",
        "minutes_played",
        "previous_weighted_goals",
        "previous_known_market_value_in_eur",
        "team_points",
        "team_final_position",
        "team_won_league",
        "player_minutes_share",
        "current_market_value_in_eur",
        "predicted_value",
        "valuation_gap",
        "has_previous_market_value",
        "minutes_band",
    ]
].head()


,player_name,season_club_name,current_club_name,position,minutes_played,previous_weighted_goals,previous_known_market_value_in_eur,team_points,team_final_position,team_won_league,player_minutes_share,current_market_value_in_eur,predicted_value,valuation_gap,has_previous_market_value,minutes_band
0,James Milner,Brighton & Hove Albion,Brighton & Hove Albion,Midfield,778,0.0,1000000.0,53,8,0,0.227485,750000.0,6.693338e+05,-8.066621e+04,True,medium_minutes
1,Casemiro,Manchester United,Manchester United,Midfield,2589,2.6,10000000.0,71,3,0,0.757018,8000000.0,6.832864e+06,-1.167136e+06,True,regular_minutes
2,Mateo Kovacic,Manchester City,Manchester City,Midfield,126,7.0,20000000.0,78,2,0,0.036842,15000000.0,1.102855e+07,-3.971451e+06,True,low_minutes
3,Jordan Henderson,Brentford FC,Brentford FC,Midfield,1921,0.7,2500000.0,53,9,0,0.561696,2000000.0,1.562267e+06,-4.377326e+05,True,regular_minutes
4,Adam Smith,AFC Bournemouth,AFC Bournemouth,Defender,1080,0.0,750000.0,57,6,0,0.315789,500000.0,6.127760e+05,1.127760e+05,True,regular_minutes


## 2025/26 valuation-gap rankings

Filter to current Premier League players with current Transfermarkt values, then rank regular-minute players by valuation gap.

In [19]:
ranking_outputs = make_ranking_tables(scoring_results)

undervalued_players = ranking_outputs["undervalued_players_2025_26"]
overvalued_players = ranking_outputs["overvalued_players_2025_26"]
high_confidence_undervalued = ranking_outputs[
    "high_confidence_undervalued_2025_26"
]
recruitment_bargains = ranking_outputs["recruitment_bargains_2025_26"]

undervalued_players

,player_name,season_club_name,current_club_name,position,sub_position,age_at_season_end,minutes_played,current_market_value_in_eur,predicted_value,valuation_gap,has_previous_market_value
237,Bukayo Saka,Arsenal FC,Arsenal FC,Attack,Right Winger,24.714579,2226,130000000.0,1.597155e+08,2.971548e+07,True
351,Florian Wirtz,Liverpool FC,Liverpool FC,Midfield,Attacking Midfield,23.058179,2390,110000000.0,1.365909e+08,2.659089e+07,True
226,Martín Zubimendi,Arsenal FC,Arsenal FC,Midfield,Defensive Midfield,27.304586,3002,50000000.0,6.891032e+07,1.891032e+07,True
207,Phil Foden,Manchester City,Manchester City,Midfield,Attacking Midfield,25.987680,2085,150000000.0,1.684482e+08,1.844817e+07,True
125,Viktor Gyökeres,Arsenal FC,Arsenal FC,Attack,Centre-Forward,27.969884,2231,70000000.0,8.470736e+07,1.470736e+07,True
212,Bryan Mbeumo,Manchester United,Manchester United,Attack,Right Winger,26.795346,2625,40000000.0,5.455024e+07,1.455024e+07,True
443,Michael Kayode,Brentford FC,Brentford FC,Defender,Right-Back,21.869952,3264,25000000.0,3.934168e+07,1.434168e+07,True
169,Declan Rice,Arsenal FC,Arsenal FC,Midfield,Central Midfield,27.356605,3099,120000000.0,1.340249e+08,1.402493e+07,True
441,Kobbie Mainoo,Manchester United,Manchester United,Midfield,Central Midfield,21.095140,1654,40000000.0,5.306862e+07,1.306862e+07,True
305,Matheus Cunha,Manchester United,Manchester United,Attack,Centre-Forward,26.992471,2503,45000000.0,5.672514e+07,1.172514e+07,True


In [20]:
overvalued_players

,player_name,season_club_name,current_club_name,position,sub_position,age_at_season_end,minutes_played,current_market_value_in_eur,predicted_value,valuation_gap,has_previous_market_value
250,Nick Woltemade,Newcastle United,Newcastle United,Attack,Centre-Forward,24.271047,1914,70000000.0,3.420483e+07,-3.579517e+07,True
329,Elliot Anderson,Nottingham Forest,Nottingham Forest,Midfield,Central Midfield,23.545517,3335,60000000.0,3.408829e+07,-2.591171e+07,True
399,Moisés Caicedo,Chelsea FC,Chelsea FC,Midfield,Defensive Midfield,24.555784,2802,110000000.0,8.763254e+07,-2.236746e+07,True
262,Reece James,Chelsea FC,Chelsea FC,Defender,Right-Back,26.458590,1960,50000000.0,2.874946e+07,-2.125054e+07,True
367,Iliman Ndiaye,Everton FC,Everton FC,Attack,Right Winger,26.214921,2789,45000000.0,2.455608e+07,-2.044392e+07,True
271,Ryan Gravenberch,Liverpool FC,Liverpool FC,Midfield,Defensive Midfield,24.021903,2996,90000000.0,6.959569e+07,-2.040431e+07,True
197,Sandro Tonali,Newcastle United,Newcastle United,Midfield,Defensive Midfield,26.042437,2545,75000000.0,5.574423e+07,-1.925577e+07,True
484,Carlos Baleba,Brighton & Hove Albion,Brighton & Hove Albion,Midfield,Defensive Midfield,22.387406,1662,60000000.0,4.150122e+07,-1.849878e+07,True
163,Cristian Romero,Tottenham Hotspur,Tottenham Hotspur,Defender,Centre-Back,28.073922,1873,60000000.0,4.176782e+07,-1.823218e+07,True
500,Josh King,Fulham FC,Fulham FC,Midfield,Attacking Midfield,19.386721,1304,20000000.0,3.000000e+06,-1.700000e+07,True


## High-confidence bargain views

Focus on regular-minute players with previous market values. This removes many new-arrival cases where the model has less historical context.

In [21]:
high_confidence_undervalued

,player_name,season_club_name,current_club_name,position,sub_position,age_at_season_end,minutes_played,previous_market_value_in_eur,current_market_value_in_eur,predicted_value,valuation_gap
237,Bukayo Saka,Arsenal FC,Arsenal FC,Attack,Right Winger,24.714579,2226,150000000.0,130000000.0,1.597155e+08,2.971548e+07
351,Florian Wirtz,Liverpool FC,Liverpool FC,Midfield,Attacking Midfield,23.058179,2390,NaN,110000000.0,1.365909e+08,2.659089e+07
226,Martín Zubimendi,Arsenal FC,Arsenal FC,Midfield,Defensive Midfield,27.304586,3002,NaN,50000000.0,6.891032e+07,1.891032e+07
207,Phil Foden,Manchester City,Manchester City,Midfield,Attacking Midfield,25.987680,2085,NaN,150000000.0,1.684482e+08,1.844817e+07
125,Viktor Gyökeres,Arsenal FC,Arsenal FC,Attack,Centre-Forward,27.969884,2231,NaN,70000000.0,8.470736e+07,1.470736e+07
212,Bryan Mbeumo,Manchester United,Manchester United,Attack,Right Winger,26.795346,2625,NaN,40000000.0,5.455024e+07,1.455024e+07
443,Michael Kayode,Brentford FC,Brentford FC,Defender,Right-Back,21.869952,3264,NaN,25000000.0,3.934168e+07,1.434168e+07
169,Declan Rice,Arsenal FC,Arsenal FC,Midfield,Central Midfield,27.356605,3099,120000000.0,120000000.0,1.340249e+08,1.402493e+07
441,Kobbie Mainoo,Manchester United,Manchester United,Midfield,Central Midfield,21.095140,1654,50000000.0,40000000.0,5.306862e+07,1.306862e+07
305,Matheus Cunha,Manchester United,Manchester United,Attack,Centre-Forward,26.992471,2503,NaN,45000000.0,5.672514e+07,1.172514e+07


In [22]:
recruitment_bargains

,player_name,season_club_name,current_club_name,position,sub_position,age_at_season_end,minutes_played,previous_market_value_in_eur,current_market_value_in_eur,predicted_value,valuation_gap
212,Bryan Mbeumo,Manchester United,Manchester United,Attack,Right Winger,26.795346,2625,NaN,40000000.0,5.455024e+07,1.455024e+07
443,Michael Kayode,Brentford FC,Brentford FC,Defender,Right-Back,21.869952,3264,NaN,25000000.0,3.934168e+07,1.434168e+07
441,Kobbie Mainoo,Manchester United,Manchester United,Midfield,Central Midfield,21.095140,1654,50000000.0,40000000.0,5.306862e+07,1.306862e+07
305,Matheus Cunha,Manchester United,Manchester United,Attack,Centre-Forward,26.992471,2503,NaN,45000000.0,5.672514e+07,1.172514e+07
481,Junior Kroupi,AFC Bournemouth,AFC Bournemouth,Attack,Centre-Forward,19.917864,1683,NaN,15000000.0,2.625402e+07,1.125402e+07
424,Igor Thiago,Brentford FC,Brentford FC,Attack,Centre-Forward,24.908966,3287,NaN,25000000.0,3.552530e+07,1.052530e+07
379,Adrien Truffert,AFC Bournemouth,AFC Bournemouth,Defender,Left-Back,24.506502,3380,NaN,20000000.0,3.039595e+07,1.039595e+07
419,Noah Sadiki,Sunderland AFC,Sunderland AFC,Midfield,Central Midfield,21.431896,2896,NaN,5000000.0,1.500000e+07,1.000000e+07
453,Jack Hinshelwood,Brighton & Hove Albion,Brighton & Hove Albion,Midfield,Defensive Midfield,21.117043,1751,24000000.0,22000000.0,3.187791e+07,9.877914e+06
420,Milos Kerkez,Liverpool FC,Liverpool FC,Defender,Left-Back,22.543463,2261,45000000.0,40000000.0,4.934368e+07,9.343682e+06


## Save ranking outputs

Save the generated 2025/26 scoring and ranking tables to `data/processed/` so they can be used by later notebooks, reports, or app views without rerunning the analysis manually.

In [23]:
save_ranking_outputs(
    ranking_outputs,
    PROCESSED_DIR,
)

,file,rows,columns
0,/Users/najib/Documents/prem-ai-valuation/data/...,537,42
1,/Users/najib/Documents/prem-ai-valuation/data/...,20,11
2,/Users/najib/Documents/prem-ai-valuation/data/...,20,11
3,/Users/najib/Documents/prem-ai-valuation/data/...,20,11
4,/Users/najib/Documents/prem-ai-valuation/data/...,20,11


## Experiment log

- Random Forest beat the selected Ridge baseline on walk-forward validation and final 2024/25 test performance.
- Adding previous Premier League performance and previous market value produced the first large improvement.
- Adding previous all-competition weighted history improved walk-forward MAE further and reduces the new-signing blind spot.
- Latest known preseason market value fixed missing-history cases such as players without a 2024/25 post-season valuation row.
- The final scoring model predicts percentage change from preseason value, then converts that change back into euros. This better matches how Transfermarkt values usually move and avoids a separate competing prediction column.
- 2025/26 scoring produced three useful ranking views: candidate undervalued players, candidate overvalued players, and a recruitment-style bargain shortlist.